In [ ]:
# Local setup: install packages once from the terminal with `python -m pip install -r requirements.txt`.


In [ ]:
# 2. Imports
import pandas as pd
from transformers import pipeline, AutoTokenizer
import numpy as np
import torch
from datasets import Dataset
from transformers.pipelines.pt_utils import KeyDataset
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "README.md").exists()
    and (PROJECT_ROOT / "requirements.txt").exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "requirements.txt").exists():
    raise FileNotFoundError(
        "Could not find the project root. Start Jupyter from the DataManagement-BusinessIntelligence folder."
    )

DATA_ROOT = PROJECT_ROOT / "Final" / "Airbnb-Full-NYC"


### **HuggingFace**

Enjoy the fun! Tons of opportunities and models to use for unstructured data:
https://huggingface.co/models

### **Load the review file**

In [ ]:
df_reviews = pd.read_csv(DATA_ROOT / "reviews-2.csv")


In [ ]:
df_reviews.shape

In [ ]:
df_reviews.head()

### **Loading Deep Learning Model**


In [ ]:
# Hyperparameter settings.

BATCH_SIZE = 192 # how many comments it will ingest at the time
MAX_LEN = 512   # safe for XLM-R; you can lower to 256 for speed if desired
MODEL_ID = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

In [ ]:
# Load tokenizer.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, model_max_length=MAX_LEN)

In [ ]:
# Making use of GPUs.
device = 0 if torch.cuda.is_available() else -1

# Loading classification pipeline.
clf = pipeline(
    task="text-classification",
    model=MODEL_ID,
    tokenizer=tokenizer,
    device=device,
    model_kwargs={"torch_dtype": "auto"}  # fp16 on GPU if available
)

In [ ]:
df_reviews.shape # takes time to process

In [ ]:
df_reviews[:500]

In [ ]:
# 5) Prepare text (fill NaNs) and run in batches for speed/memory safety
texts = df_reviews["comments"].fillna("").astype(str).tolist()
df_reviews["comments"] = df_reviews["comments"].fillna("").astype(str)

################################################################
#################### PLEASE READ BELOW #########################
################################################################
# As is, we will analyze only the first 500 comments for time limits.
# If you want to classify all the comments, please comment line 10 -- put an hashtag, it has to run green
df_reviews = df_reviews[:500]

# 2) Build Dataset with a 'text' column (what pipelines expect if not raw strings)
ds = Dataset.from_pandas(df_reviews[["comments"]]).remove_columns(["__index_level_0__"]) if "__index_level_0__" in df_reviews.columns else Dataset.from_pandas(df_reviews[["comments"]])
ds = ds.rename_column("comments", "text")


In [ ]:
texts[:3]

In [ ]:
# 3) Stream to the pipeline
pred_iter = clf(
    KeyDataset(ds, "text"),
    batch_size=BATCH_SIZE,
    top_k=None,          # gets all class scores
    truncation=True,
    padding=True,
    max_length=MAX_LEN,
)

In [ ]:
# Collect predictions
raw = list(pred_iter)  # each item: [{'label':'negative','score':...}, {'label':'neutral',...}, {'label':'positive',...}]

In [ ]:
raw[:10]

In [ ]:
labels = ["negative", "neutral", "positive"]

probs = np.array([
    [next(s["score"] for s in row if s["label"] == lab) for lab in labels]
    for row in raw
])

In [ ]:
df_reviews["sentiment_neg_prob"] = probs[:, 0]
df_reviews["sentiment_neu_prob"] = probs[:, 1]
df_reviews["sentiment_pos_prob"] = probs[:, 2]
df_reviews["sentiment_label"] = [labels[i] for i in probs.argmax(axis=1)]
df_reviews["sentiment_confidence"] = probs.max(axis=1)

In [ ]:
df_reviews

In [ ]:
np.where(df_reviews["sentiment_label"] == "negative")[0]

In [ ]:
df_reviews.iloc[30].comments

In [ ]:
df_reviews.iloc[496].comments

In [ ]:
# Save the DataFrame locally in the project data folder.
df_reviews.to_csv(DATA_ROOT / "reviews_with_sentiment.csv", index=False)
print("DataFrame saved locally.")
